In [1]:
import os
import pandas as pd

### Concatenate all results dataframes

In [2]:
def generate_full_results_df(src_path, model_name):

    # accuracy
    path = os.path.join(src_path, "cl_metrics/accuracy_matrix_mean_std.csv")
    df_acc = pd.read_csv(path)
    df_acc[['fomc','fomc_std']] = df_acc['fomc'].str.split(' ± ',expand=True)
    df_acc[['scienceqa','scienceqa_std']] = df_acc['scienceqa'].str.split(' ± ',expand=True)
    df_acc[['numgluecm','numgluecm_std']] = df_acc['numgluecm'].str.split(' ± ',expand=True)

    # knowledge acquisition
    path = os.path.join(src_path, "cl_metrics/table_acquisition.csv")
    df_acquisition = pd.read_csv(path)
    df_acquisition[['curr_accuracy','curr_accuracy_std']] = df_acquisition['Current accuracy'].str.split(' ± ',expand=True)
    df_acquisition[['delta_curr_vs_base','delta_curr_vs_base_std']] = df_acquisition['Δ current vs base'].str.split(' ± ',expand=True)
    df_acquisition[['avg_over_all_tasks','avg_over_all_tasks_std']] = df_acquisition['Average over all tasks'].str.split(' ± ',expand=True)
    df_acquisition.drop(columns=["Current accuracy", "Δ current vs base", "Average over all tasks"], inplace=True)
    df_acquisition.rename(columns={"Checkpoint": "checkpoint", "Step": "step"}, inplace=True)

    # stability
    path = os.path.join(src_path, "stability_metrics/table_stability.csv")
    df_stability = pd.read_csv(path)
    df_stability[['entropy_mean','entropy_mean_std']] = df_stability['entropy_mean'].str.split(' ± ',expand=True)
    df_stability[['margin_mean','margin_mean_std']] = df_stability['margin_mean'].str.split(' ± ',expand=True)
    df_stability[['kl_to_base_mean','kl_to_base_mean_std']] = df_stability['kl_to_base_mean'].str.split(' ± ',expand=True)
    df_stability.drop(columns=["entropy_p95", "margin_p95", "kl_to_base_p95"], inplace=True)
    
    # forgetting
    path =  os.path.join(src_path, "cl_metrics/table_transfer_forgetting.csv")
    df_forgetting = pd.read_csv(path)
    df_forgetting[['op','op_std']] = df_forgetting['OP'].str.split(' ± ',expand=True)
    df_forgetting[['bwt','bwt_std']] = df_forgetting['BWT'].str.split(' ± ',expand=True)
    df_forgetting[['fwt','fwt_std']] = df_forgetting['FWT'].str.split(' ± ',expand=True)
    df_forgetting.drop(columns=["OP", "BWT", "FWT", "Forgetting", "Next-task FWT"], inplace=True)
    df_forgetting.rename(columns={"Checkpoint": "checkpoint", "Step": "step", "Next task": "next_task"}, inplace=True)

    # merging results
    df_merged = pd.merge(left=df_acc, right=df_stability, on=["step", "checkpoint"])
    df_merged = pd.merge(left=df_merged, right=df_acquisition, on=["step", "checkpoint"])
    df_merged = pd.merge(left=df_merged, right=df_forgetting, on=["step", "checkpoint"])
    df_merged.insert(0, "model", model_name)
    
    return df_merged

In [3]:
def generate_all_results(config):
    dfs = []
    for model, path in config.items():
        df = generate_full_results_df(path, model)
        dfs.append(df)

    return pd.concat(dfs)

In [5]:
config = {
    "qwen": "../results/qwen35_08b_paper",
    "gemma": "../results/gemma3_1b_paper/",
    "llama": "../results/llama32_1b_paper/"
}
df_all = generate_all_results(config)
df_all.head()

,model,step,checkpoint,checkpoint_task,fomc,numgluecm,scienceqa,fomc_std,scienceqa_std,numgluecm_std,...,delta_curr_vs_base_std,avg_over_all_tasks,avg_over_all_tasks_std,next_task,op,op_std,bwt,bwt_std,fwt,fwt_std
0,qwen,0,Base,base,0.250,0.370,0.520,0.000,0.000,0.000,...,NaN,0.380,0.000,task_1,—,NaN,—,NaN,0.000,0.000
1,qwen,1,After task_1,task_1,0.533,0.379,0.660,0.076,0.035,0.019,...,0.076,0.524,0.029,task_2,0.533,0.076,—,NaN,0.074,0.012
2,qwen,2,After task_2,task_2,0.540,0.490,0.800,0.053,0.000,0.026,...,0.000,0.610,0.012,task_3,0.670,0.026,0.007,0.075,0.119,0.026
3,qwen,3,After task_3,task_3,0.577,0.379,0.817,0.042,0.006,0.056,...,0.056,0.591,0.012,—,0.591,0.012,0.030,0.023,—,NaN
0,gemma,0,Base,base,0.220,0.370,0.620,0.000,0.000,0.000,...,NaN,0.403,0.000,task_1,—,NaN,—,NaN,0.000,0.000


In [6]:
df_all.to_csv("combined_df_v2.csv")

In [7]:
config = {
    "qwen": "../results/qwen35_08b_paper_321/",
    "gemma": "../results/gemma3_1b_paper_321/",
    "llama": "../results/llama32_1b_paper_321/"
}
df_all_321 = generate_all_results(config)
df_all_321.head()

,model,step,checkpoint,checkpoint_task,fomc,numgluecm,scienceqa,fomc_std,scienceqa_std,numgluecm_std,...,delta_curr_vs_base_std,avg_over_all_tasks,avg_over_all_tasks_std,next_task,op,op_std,bwt,bwt_std,fwt,fwt_std
0,qwen,0,Base,base,0.250,0.370,0.520,0.000,0.000,0.000,...,NaN,0.380,0.000,task_3,—,NaN,—,NaN,0.000,0.000
1,qwen,1,After task_3,task_3,0.287,0.185,0.693,0.012,0.035,0.098,...,0.098,0.388,0.035,task_2,0.185,0.098,—,NaN,0.105,0.018
2,qwen,2,After task_2,task_2,0.287,0.235,0.817,0.029,0.012,0.037,...,0.012,0.446,0.007,task_1,0.526,0.014,0.049,0.074,0.037,0.029
3,qwen,3,After task_1,task_1,0.620,0.255,0.820,0.035,0.010,0.056,...,0.035,0.565,0.020,—,0.565,0.020,0.037,0.026,—,NaN
0,gemma,0,Base,base,0.220,0.370,0.620,0.000,0.000,0.000,...,NaN,0.403,0.000,task_3,—,NaN,—,NaN,0.000,0.000


In [8]:
df_all_321.to_csv("combined_df_321_v2.csv")